# Transaction Foundation Model on Ray — Part 3: Tokenize — flatten transactions into tokens

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 2 we built the canonical dataset and the temporal split for evaluation. Now, tokenize — we turn each card's raw transactions into the flat token sequences the decoder trains on (NVIDIA's ~12-tokens-per-transaction scheme), as a stateless `groupby("card_id").map_groups(...)` that runs across the cluster, one card at a time.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir, stale_or_missing
from src.scale_config import load_scale

SCALE = "full"                                        # same knob as Part 2; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

with open(paths["splits"]) as f:
    splits = json.load(f)                            # temporal cutoffs from Part 2

## Flattening transactions into tokens

We follow NVIDIA's transaction-FM blueprint: **every transaction becomes ~12 tokens** in one shared vocabulary, and a card's history is a flat token stream a decoder reads left-to-right.

Each transaction contributes 12 field tokens in a fixed order — `AMT MERCH CAT MCC HOUR DOW MONTH CARD CHIP ZIP3 STATE CUST` — drawn from a single ~6.3k-token vocabulary (each field owns a disjoint id range). The card's sequence is then `<bos> [txn₁ tokens] <sep> [txn₂ tokens] <sep> … <eos>`, padded to `seq_len` **tokens** (so `full`'s 4096 tokens ≈ 314 transactions of context). Bounded fields (hour, day, month, card index, ZIP3, customer id) map to fixed ranges; open-ended ones (merchant, MCC) are hashed. The vocabulary is fully deterministic — no data scan, no fit — so tokenization stays a stateless, per-card `map_groups`.

This flat stream is what a **Llama causal decoder** pretrains on by next-token prediction (Part 4).

## Turn each card's history into token sequences

We now turn each card's raw transactions into the flat integer token sequences the decoder trains on. This writes two datasets: pretraining sequences and labeled eval samples.

We group transactions by `card_id` and run a tokenization function per card with `map_groups`, since a card's tokens depend on its whole ordered history. Because the tokenizer is stateless, this scales trivially with Ray.

Two configuration parameters:

- `normal_keep` comes from `eval_normal_keep`: it keeps every fraud and downsamples normals toward `target_eval_samples`. Each kept normal is stamped with a weight of `1 / normal_keep` so the downsampling cancels out in the metrics (Part 6).
- `num_partitions` sizes the group-by shuffle. Ray defaults to 200, which would scatter the output into hundreds of tiny files at this scale.

In one pass the tokenizer returns two kinds of row per card, distinguished by a `kind` column:
- **pretrain** — the card's train-period history packed into token sequences, for next-token pretraining (Part 4); never includes val/test transactions.
- **eval** — one window per target transaction (ending at it), labeled with that transaction's `is_fraud` and tagged `train`/`val`/`test` by its timestamp.

After `materialize`, the cell filters on `kind` and writes the two sets to separate Parquet directories (`PRETRAIN_DROP` strips the eval-only columns from the pretrain set), then `write_vocab` records the fixed vocabulary spec the model reads.

In [ ]:
from ray.data.expressions import col
from src.tokenizer import tokenize_dataset, eval_normal_keep, write_vocab, PRETRAIN_DROP

tk = cfg["tokenize"]
seq_len = tk["seq_len"]                                # NOW IN TOKENS (~12 per transaction)
# train_keep sizes the downstream TRAINING set (full = 1.0 -> train on the whole
# training period, like NVIDIA); holdout_keep sizes val/test (1.0 = score every
# holdout txn for exact metrics, else fall back to the target_eval_samples rate).
train_keep = tk["train_keep"]
holdout_keep = (tk["holdout_keep"] if tk["holdout_keep"] is not None
                else eval_normal_keep(splits, tk["target_eval_samples"]))
print(f"tokenizing at seq_len={seq_len} tokens, train_keep={train_keep:.4f} holdout_keep={holdout_keep:.4f}")

# Re-runs reuse the cached token shards — but only if they're CURRENT. stale_or_missing
# rebuilds when any token artifact is absent OR older than the raw inputs it derives
# from, so a regenerated dataset can never be silently served from a stale cache
# (the skip is content-aware, not just "does the directory exist").
_tok_inputs = [paths["raw"], paths["splits"]]
_tok_outputs = [paths["tokenized_pretrain"], paths["tokenized_eval"], paths["vocab"]]
if any(stale_or_missing(out, _tok_inputs) for out in _tok_outputs):
    # One card = one group. tokenize_dataset frames each card's history as a
    # <bos> ... <sep> ... <eos> stream of ~12-token transactions (NVIDIA's scheme);
    # scripts/02_tokenize.py composes the exact same call so walkthrough and job
    # can't drift. We STREAM each emit straight to Parquet rather than materialize
    # the combined ~24M-window set (that would spill hundreds of GB).
    def tokenized(emit):
        return tokenize_dataset(
            ray.data.read_parquet(paths["raw"]), seq_len,
            train_end=splits["train_end"], val_end=splits["val_end"],
            normal_keep=train_keep, holdout_keep=holdout_keep,
            max_pretrain_windows=tk["max_pretrain_windows"],
            num_partitions=tk["shuffle_partitions"], emit=emit,
        )

    # Pretrain: non-overlapping train-period windows (input_ids/attention_mask only).
    tokenized("pretrain").filter(expr=col("kind") == "pretrain") \
        .drop_columns(PRETRAIN_DROP).write_parquet(paths["tokenized_pretrain"])
    # Eval: one window per target transaction, carrying label / split / raw_* features.
    tokenized("eval").filter(expr=col("kind") == "eval") \
        .drop_columns(["kind"]).write_parquet(paths["tokenized_eval"])
    write_vocab(paths["vocab"], seq_len)              # deterministic flat-vocab spec the model reads

    print(f"    pretrain -> {paths['tokenized_pretrain']}")
    print(f"    eval     -> {paths['tokenized_eval']}")
    print(f"    vocab    -> {paths['vocab']}")
else:
    print(f"  reusing cached tokens at {os.path.dirname(paths['tokenized_eval'].rstrip('/'))}/ (current with raw data)")


## The tokenized output

This yields the pretrain sequences (one packed stream of a card's train-period history) and the eval samples — one per target transaction, each **every fraud kept** plus downsampled normals, tagged `train`/`val`/`test` by the temporal split. Each normal eval sample carries a weight of `1 / normal_keep` so the downsampling cancels out in the metrics; frauds carry weight 1.

One eval sample makes it concrete. We decode its flat `input_ids` back to readable tokens — each is an id into the shared vocabulary (`merchant`/`mcc` are hashed, so they show as bucket numbers). The 12 tokens per transaction appear in the fixed field order, framed by `<bos>`/`<sep>`/`<eos>`.

In [3]:
import glob
import pyarrow.parquet as pq
from src.tokenizer import decode_tokens, TOKENS_PER_TXN

def _parquet_rows(path):
    """Row count straight from the Parquet footers — no data scan."""
    return sum(pq.ParquetFile(f).metadata.num_rows
               for f in glob.glob(path.rstrip("/") + "/*.parquet"))

# Counts from Parquet metadata; the split/fraud breakdown reads only the two
# scalar columns it touches (projection pushdown skips the wide token arrays).
n_pre = _parquet_rows(paths["tokenized_pretrain"])
meta = pd.read_parquet(paths["tokenized_eval"], columns=["split", "label"])
by = meta["split"].value_counts()

print(f"{n_pre:,} pretrain sequences")
print(f"{len(meta):,} eval samples  ({int((meta['label'] == 1).sum()):,} fraud · "
      f"train {by.get('train', 0):,} / val {by.get('val', 0):,} / test {by.get('test', 0):,})")
print(f"seq_len {seq_len} tokens  (~{TOKENS_PER_TXN} tokens/transaction)\n")

# Decode one eval sample's flat token stream back to readable tokens. Read a
# single shard, take its first row — not the whole eval set. (input_ids comes
# back as a pyarrow tensor element, so wrap it in np.asarray to iterate.)
one_shard = sorted(glob.glob(paths["tokenized_eval"].rstrip("/") + "/*.parquet"))[0]
row = pd.read_parquet(one_shard).iloc[0]
ids = [int(t) for t in np.asarray(row["input_ids"]) if int(t) != 0]   # drop right-padding
toks = decode_tokens(ids)
print(f"One eval sample — card {int(row['card_id'])}, label {int(row['label'])}, {len(ids)} tokens\n")
print("Flat token stream (first ~2 transactions):")
print("  " + " ".join(toks[: 1 + 2 * (TOKENS_PER_TXN + 1)]))
print("\nOrder per transaction: AMT MERCH CAT MCC HOUR DOW MONTH CARD CHIP ZIP3 STATE CUST")

65,981 pretrain sequences
24,386,900 eval samples  (29,757 fraud · train 19,509,517 / val 2,438,693 / test 2,438,690)
seq_len 4096 tokens  (~12 tokens/transaction)

One eval sample — card 1702, label 0, 14 tokens

Flat token stream (first ~2 transactions):
  <bos> AMT_1 MERCH_1868 CAT_0 MCC_75 HOUR_12 DOW_1 MONTH_0 CARD_2 CHIP_0 ZIP3_860 STATE_2 CUST_17 <eos>

Order per transaction: AMT MERCH CAT MCC HOUR DOW MONTH CARD CHIP ZIP3 STATE CUST


The vocabulary is fixed, so the model knows its size up front — no data scan, no cold-start. Specials (`<pad>`/`<bos>`/`<eos>`/`<sep>`/`<unk>`) come first, then each field owns a disjoint block of ids (7 amount bins, 2000 hashed merchant buckets, 24 hours, and so on), summing to the shared vocab.

In [4]:
from src.tokenizer import FIELD_SPECS, TOKENS_PER_TXN, VOCAB_SIZE, SPECIALS

print(f"shared vocabulary: {VOCAB_SIZE:,} tokens  ({len(SPECIALS)} special + per-field blocks)")
print(f"{TOKENS_PER_TXN} tokens per transaction\n")
print("per-field block size in the shared vocab:")
for name, size in FIELD_SPECS:
    print(f"  {name:<7} {size:>6,}")

shared vocabulary: 6,259 tokens  (5 special + per-field blocks)
12 tokens per transaction

per-field block size in the shared vocab:
  AMT          7
  MERCH    2,000
  CAT          9
  MCC        128
  HOUR        24
  DOW          7
  MONTH       12
  CARD        10
  CHIP         3
  ZIP3     1,000
  STATE       54
  CUST     3,000


## Takeaways

**Ray Data**
- Tokenization is a `groupby("card_id").map_groups(fn)`: one card is one group, and `fn` needs the whole ordered history to build its token stream.
- Because the vocabulary is deterministic, `fn` is **stateless** — no global aggregation, no cross-card coordination, so every group tokenizes independently across the cluster. The same code runs on one CPU at `mini` and across the cluster at `full`.
- `num_partitions` right-sizes the hash shuffle; Ray's default (200) is tuned for far larger clusters and would scatter the output into hundreds of tiny files at demo scale.
- The notebook composes the same `make_tokenize_group_fn` that `scripts/02_tokenize.py` runs headless, so the walkthrough and the batch job produce identical tokens.

**The tokenizer**
- NVIDIA's flat scheme: each transaction → ~12 tokens in one shared vocabulary; a card's history is a `<bos> … <sep> … <eos>` stream, padded to `seq_len` tokens.
- One pass emits two streams: train-period **pretrain** sequences (for next-token pretraining) and per-transaction **eval** samples (all fraud kept, normals downsampled with importance weights).
- Bounded fields map to fixed id ranges; open-ended ones (merchant, MCC) are hashed — fully deterministic, no fit.

---

## Next

**Part 4 — Pretrain with Ray Train**: next-token prediction over the pretrain sequences with a Llama causal decoder, plain PyTorch wrapped by Ray Train for DDP, sharding, and checkpointing — same code from 1 CPU worker to N GPUs.